<a href="https://colab.research.google.com/github/shanmughamkarthik-eng/SQL_Assignment/blob/main/Visitors_SQL_assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
-- 1. Create CUSTOMERS Table
CREATE TABLE CUSTOMERS (
    CUSTOMER_ID NUMBER PRIMARY KEY,
    FIRST_NAME VARCHAR2(50),
    LAST_NAME VARCHAR2(50),
    CITY VARCHAR2(50),
    PHONE_NUMBER VARCHAR2(20),
    LOYALTY_POINTS NUMBER
);

-- 2. Create ORDERS Table
CREATE TABLE ORDERS (
    ORDER_ID NUMBER PRIMARY KEY,
    CUSTOMER_ID NUMBER,
    ORDER_DATE DATE,
    TOTAL_AMOUNT NUMBER(10, 2),
    DISCOUNT_AMT NUMBER(10, 2), -- Can be NULL if no discount
    SHIPPING_DATE DATE          -- Can be NULL if not shipped yet
);

-- 3. Insert Data with intentional NULL values
-- Customers
INSERT INTO CUSTOMERS VALUES (101, 'John', 'Doe', 'New York', '555-0100', 500);
INSERT INTO CUSTOMERS VALUES (102, 'Jane', 'Smith', NULL, '555-0101', 120);
INSERT INTO CUSTOMERS VALUES (103, 'Robert', 'Brown', 'Chicago', NULL, 0);
INSERT INTO CUSTOMERS VALUES (104, 'Emily', 'Davis', NULL, NULL, NULL); -- Lots of NULLs
INSERT INTO CUSTOMERS VALUES (105, 'Michael', 'Wilson', 'Miami', '555-0105', NULL);

-- Orders
INSERT INTO ORDERS VALUES (5001, 101, DATE '2023-10-01', 150.00, 10.00, DATE '2023-10-03');
INSERT INTO ORDERS VALUES (5002, 102, DATE '2023-10-02', 200.50, NULL, DATE '2023-10-05'); -- No discount
INSERT INTO ORDERS VALUES (5003, 101, DATE '2023-10-05', 75.00, 5.00, NULL); -- Not shipped
INSERT INTO ORDERS VALUES (5004, 104, DATE '2023-10-06', 300.00, NULL, NULL); -- No discount, Not shipped
INSERT INTO ORDERS VALUES (5005, 105, DATE '2023-10-07', 50.00, 0.00, DATE '2023-10-08');
INSERT INTO ORDERS VALUES (5006, NULL, DATE '2023-10-08', 20.00, NULL, DATE '2023-10-09'); -- Orphan order

COMMIT;

Here are 40 questions focused exclusively on **SQL Joins** (Inner, Left, Right, Full, Cross, Self, and Natural). These questions are designed to test how Joins interact with **NULL** values in your `CUSTOMERS` and `ORDERS` tables.

### Part 1: 20 Solved Join Questions (with Answers)

**1. (Inner Join) Retrieve a list of customers who have placed at least one order. Display Name and Order ID.**
*Note: This excludes customers with no orders and orders with NULL Customer IDs.*

 ```sql
SELECT C.FIRST_NAME, C.LAST_NAME, O.ORDER_ID
FROM CUSTOMERS C
INNER JOIN ORDERS O
ON C.CUSTOMER_ID = O.CUSTOMER_ID;
```


      FIRST_NAME LAST_NAME ORDER_ID
      ---------- --------- --------
      John       Doe       5001     
      Jane       Smith     5002     
      John       Doe       5003     
      Emily      Davis     5004     
      Michael    Wilson    5005     

      Elapsed: 00:00:00.001
      5 rows selected.



**2. (Left Join) List all customers and their Order IDs. Include customers who have not placed any orders.**
*Note: Returns NULL in the Order column for customers like 'Robert' or 'Emily'.*

```sql
SELECT C.FIRST_NAME, O.ORDER_ID
FROM CUSTOMERS C
LEFT JOIN ORDERS O
ON C.CUSTOMER_ID = O.CUSTOMER_ID;

      FIRST_NAME ORDER_ID
      ---------- --------
      John       5001     
      Jane       5002     
      John       5003     
      Emily      5004     
      Michael    5005     
      Robert              

      Elapsed: 00:00:00.016
      6 rows selected.



**3. (Left Join - Finding Non-Matches) Find customers who have NEVER placed an order.**
*Note: We join, then filter for where the "Right" table is NULL.*
```sql
SELECT C.FIRST_NAME, C.LAST_NAME
FROM CUSTOMERS C
LEFT JOIN ORDERS O
ON C.CUSTOMER_ID = O.CUSTOMER_ID
WHERE O.ORDER_ID IS NULL;
```

      FIRST_NAME LAST_NAME
      ---------- ---------
      Robert     Brown     

      Elapsed: 00:00:00.007
      1 rows selected.



**4. (Right Join) List all orders and the associated customer name. Include orders that do not have a linked Customer ID.**
*Note: This will show Order 5006, which has a NULL Customer_ID.*
```sql
SELECT C.FIRST_NAME, O.ORDER_ID, O.TOTAL_AMOUNT
FROM CUSTOMERS C
RIGHT JOIN ORDERS O
ON C.CUSTOMER_ID = O.CUSTOMER_ID;
```

      FIRST_NAME ORDER_ID TOTAL_AMOUNT
      ---------- -------- ------------
      John       5001     150          
      John       5003     75           
      Jane       5002     200.5        
      Emily      5004     300          
      Michael    5005     50           
                5006     20           

      Elapsed: 00:00:00.114
      6 rows selected.



**5. (Full Outer Join) List ALL customers and ALL orders. If a customer has no order, show NULLs for order info. If an order has no customer, show NULLs for customer info.**
```sql
SELECT C.FIRST_NAME, O.ORDER_ID
FROM CUSTOMERS C
FULL OUTER JOIN ORDERS O
ON C.CUSTOMER_ID = O.CUSTOMER_ID;
```

      FIRST_NAME ORDER_ID
      ---------- --------
      John       5001     
      Jane       5002     
      John       5003     
      Emily      5004     
      Michael    5005     
                5006     
      Robert              

      Elapsed: 00:00:00.005
      7 rows selected.



**6. (Left Join with NVL) Calculate the total amount spent by each customer. If they haven't bought anything, display 0 instead of NULL.**
```sql
SELECT C.FIRST_NAME, NVL(SUM(O.TOTAL_AMOUNT), 0) AS TOTAL_SPENT
FROM CUSTOMERS C
LEFT JOIN ORDERS O
ON C.CUSTOMER_ID = O.CUSTOMER_ID
GROUP BY C.FIRST_NAME;
```

      FIRST_NAME TOTAL_SPENT
      ---------- -----------
      John       225         
      Jane       200.5       
      Emily      300         
      Michael    50          
      Robert     0           

      Elapsed: 00:00:00.010
      5 rows selected.


**7. (Self Join) Find pairs of customers who live in the same City.**
*Note: We must exclude NULL cities and ensure we don't match a customer to themselves.*
```sql
SELECT A.FIRST_NAME AS CUST_1, B.FIRST_NAME AS CUST_2, A.CITY
FROM CUSTOMERS A
JOIN CUSTOMERS B ON A.CITY = B.CITY
WHERE A.CUSTOMER_ID < B.CUSTOMER_ID;
```


      0 rows selected.



**8. (Cross Join) Generate a theoretical list of every customer buying every order (Cartesian Product).**
```sql
SELECT C.FIRST_NAME, O.ORDER_ID
FROM CUSTOMERS C
CROSS JOIN ORDERS O;
```

      FIRST_NAME ORDER_ID
      ---------- --------
      John       5001     
      John       5002     
      John       5003     
      John       5004     
      John       5005     
      John       5006     
      Jane       5001     
      Jane       5002     
      Jane       5003     
      Jane       5004     
      Jane       5005     
      Jane       5006     
      Robert     5001     
      Robert     5002     
      Robert     5003     
      Robert     5004     
      Robert     5005     
      Robert     5006     
      Emily      5001     
      Emily      5002     
      Emily      5003     
      Emily      5004     
      Emily      5005     
      Emily      5006     
      Michael    5001     
      Michael    5002     
      Michael    5003     
      Michael    5004     
      Michael    5005     
      Michael    5006     

      Elapsed: 00:00:00.006
      30 rows selected.



**9. (Natural Join) Join Customers and Orders automatically based on the common column (`CUSTOMER_ID`).**
*Note: Oracle does not require table aliases on the common column in a Natural Join.*
```sql
SELECT FIRST_NAME, ORDER_ID
FROM CUSTOMERS
NATURAL JOIN ORDERS;
```

      FIRST_NAME ORDER_ID
      ---------- --------
      John       5001     
      Jane       5002     
      John       5003     
      Emily      5004     
      Michael    5005     

      Elapsed: 00:00:00.016
      5 rows selected.

**10. (Left Join & Filtering) List all Customers and their Order Dates, but only for orders placed after October 5th. Keep customers with NO orders in the list.**
*Note: The condition must be in the `ON` clause, not `WHERE`, to preserve the Left Join behavior for non-matching rows.*
```sql
SELECT C.FIRST_NAME, O.ORDER_DATE
FROM CUSTOMERS C
LEFT JOIN ORDERS O ON C.CUSTOMER_ID = O.CUSTOMER_ID AND O.ORDER_DATE > DATE '2023-10-05';
```


      FIRST_NAME ORDER_DATE                
      ---------- -------------------------
      Emily      10/06/2023, 05:30:00 AM   
      Michael    10/07/2023, 05:30:00 AM   
      John                                 
      Robert                               
      Jane                                 

      Elapsed: 00:00:00.011
      5 rows selected.




**11. (Full Join - Exclusive) Find records that are EITHER a Customer without an order OR an Order without a Customer (The "Anti-Join" of both sides).**
```sql
SELECT C.CUSTOMER_ID AS C_ID, O.ORDER_ID
FROM CUSTOMERS C
FULL OUTER JOIN ORDERS O ON C.CUSTOMER_ID = O.CUSTOMER_ID
WHERE C.CUSTOMER_ID IS NULL OR O.CUSTOMER_ID IS NULL;
```

      C_ID ORDER_ID
      ---- --------
          5006     
      103           

      Elapsed: 00:00:00.005
      2 rows selected.


**12. (Inner Join with Aggregates) Find the average loyalty points of customers who have actually placed an order.**
*Note: Distinct ensures we don't double count points if a customer ordered twice.*
```sql
SELECT AVG(DISTINCT C.LOYALTY_POINTS)
FROM CUSTOMERS C
JOIN ORDERS O ON C.CUSTOMER_ID = O.CUSTOMER_ID;
```

      AVG(DISTINCTC.LOYALTY_POINTS)
      -----------------------------
      310                           

      Elapsed: 00:00:00.009
      1 rows selected.



**13. (Left Join with Coalesce) Display a report: "Customer Name - Order ID". If there is no order, display "Customer Name - No Order".**
```sql
SELECT C.FIRST_NAME || ' - ' || COALESCE(TO_CHAR(O.ORDER_ID), 'No Order') AS REPORT_LOG
FROM CUSTOMERS C
LEFT JOIN ORDERS O ON C.CUSTOMER_ID = O.CUSTOMER_ID;
```

      REPORT_LOG          
      -------------------
      John - 5001         
      Jane - 5002         
      John - 5003         
      Emily - 5004        
      Michael - 5005      
      Robert - No Order   

      Elapsed: 00:00:00.005
      6 rows selected.




**14. (Self Join on Inequality) Find pairs of orders where the first order has a higher amount than the second order.**
```sql
SELECT O1.ORDER_ID AS HIGH_VAL_ORDER, O2.ORDER_ID AS LOW_VAL_ORDER
FROM ORDERS O1
JOIN ORDERS O2 ON O1.TOTAL_AMOUNT > O2.TOTAL_AMOUNT;
```


      HIGH_VAL_ORDER LOW_VAL_ORDER
      -------------- -------------
      5004           5002          
      5004           5001          
      5004           5003          
      5004           5005          
      5004           5006          
      5002           5001          
      5002           5003          
      5002           5005          
      5002           5006          
      5001           5003          
      5001           5005          
      5001           5006          
      5003           5005          
      5003           5006          
      5005           5006          

      Elapsed: 00:00:00.009
      15 rows selected.



**15. (Left Join) Count how many orders each customer has made.**
*Note: `COUNT(O.ORDER_ID)` is safer than `COUNT(*)` because it returns 0 for NULLs.*
```sql
SELECT C.FIRST_NAME, COUNT(O.ORDER_ID) AS ORDER_COUNT
FROM CUSTOMERS C
LEFT JOIN ORDERS O ON C.CUSTOMER_ID = O.CUSTOMER_ID
GROUP BY C.FIRST_NAME;
```


      FIRST_NAME ORDER_COUNT
      ---------- -----------
      John       2           
      Jane       1           
      Emily      1           
      Michael    1           
      Robert     0           

      Elapsed: 00:00:00.008
      5 rows selected.




**16. (Right Join with Date Logic) List all orders and the Customer City. If the Customer is NULL (orphan order), display 'Unknown City'.**
```sql
SELECT O.ORDER_ID, NVL(C.CITY, 'Unknown City')
FROM CUSTOMERS C
RIGHT JOIN ORDERS O ON C.CUSTOMER_ID = O.CUSTOMER_ID;
```


      ORDER_ID NVL(C.CITY,'UNKNOWNCITY')
      -------- -------------------------
      5001     New York                  
      5003     New York                  
      5002     Unknown City              
      5004     Unknown City              
      5005     Miami                     
      5006     Unknown City              

      Elapsed: 00:00:00.005
      6 rows selected.




**17. (Non-Equi Join) Find customers whose Loyalty Points are greater than the Total Amount of any single order.**
*Note: Joining based on value comparison rather than ID equality.*
```sql
SELECT DISTINCT C.FIRST_NAME, C.LOYALTY_POINTS, O.TOTAL_AMOUNT
FROM CUSTOMERS C
JOIN ORDERS O ON C.LOYALTY_POINTS > O.TOTAL_AMOUNT;
```


      FIRST_NAME LOYALTY_POINTS TOTAL_AMOUNT
      ---------- -------------- ------------
      John       500            300          
      John       500            200.5        
      John       500            150          
      John       500            75           
      John       500            50           
      John       500            20           
      Jane       120            75           
      Jane       120            50           
      Jane       120            20           

      Elapsed: 00:00:00.012
      9 rows selected.




**18. (Left Join) List Customers and their "Shipping Status". If the order exists but Shipping Date is NULL, show 'Pending'. If no order exists, show 'N/A'.**
```sql
SELECT C.FIRST_NAME,
       CASE
           WHEN O.ORDER_ID IS NULL THEN 'N/A'
           WHEN O.SHIPPING_DATE IS NULL THEN 'Pending'
           ELSE 'Shipped'
       END AS STATUS
FROM CUSTOMERS C
LEFT JOIN ORDERS O ON C.CUSTOMER_ID = O.CUSTOMER_ID;
```


      FIRST_NAME STATUS    
      ---------- ---------
      John       Shipped   
      Jane       Shipped   
      John       Pending   
      Emily      Pending   
      Michael    Shipped   
      Robert     N/A       

      Elapsed: 00:00:00.007
      6 rows selected.




**19. (Using USING) Perform an Inner Join using the `USING` clause instead of `ON`.**
*Note: This works because both tables share the column name `CUSTOMER_ID`.*
```sql
SELECT FIRST_NAME, ORDER_ID
FROM CUSTOMERS
JOIN ORDERS USING (CUSTOMER_ID);
```


      FIRST_NAME ORDER_ID
      ---------- --------
      John       5001     
      Jane       5002     
      John       5003     
      Emily      5004     
      Michael    5005     

      Elapsed: 00:00:00.009
      5 rows selected.




**20. (Multiple Join Conditions) Join Customers and Orders where the Customer ID matches AND the Customer has a valid City (City is not null).**
```sql
SELECT C.FIRST_NAME, O.ORDER_ID
FROM CUSTOMERS C
JOIN ORDERS O ON C.CUSTOMER_ID = O.CUSTOMER_ID
WHERE C.CITY IS NOT NULL;
```


      FIRST_NAME ORDER_ID
      ---------- --------
      John       5001     
      John       5003     
      Michael    5005     

      Elapsed: 00:00:00.007
      3 rows selected.

